# 09 — Input current and efficiency

The input source sees $I_{\mathrm{IN}} = I_{\mathrm{OUT}} + I_{\mathrm{GND}}$. At 12 V in and 50 mA out, a 5 V linear regulator is about 42 % efficient — expected.

## Setup — editable BOM and run knobs

This cell is shared by every notebook. It puts `P5V_Regulator/` on `sys.path`
and builds **one** `CircuitParams` object. Change a resistor, capacitor, or
tolerance here and re-run: analytical, SPICE, WC, and MC all follow this object.

**Plot callouts:** each figure cell has an `ann = [dict(...), ...]` list.
Edit `text`, `xy` (data coords of the arrow tip), and `offset=(dx, dy)` in
**points** to nudge a label. Use `xytext=(x, y)` instead of `offset` for data
coordinates. `arrow=False` for a plain label. `ann = []` hides all callouts.

Tolerances are **fractions** (0.01 = 1 %). Figures go to `results/figures/`.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _find_project_root() -> Path:
    start = Path.cwd().resolve()
    named = Path("Sections/Electronics/Analog_and_Digital/LT3010_5V_Supply")
    candidates = [start, *start.parents, start / named]
    if start.name == "notebooks":
        candidates.insert(0, start.parent)
    for cand in candidates:
        if (cand / "p5v_design" / "__init__.py").is_file():
            return cand
    return start

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from p5v_design.params import CircuitParams, ToleranceSpec, format_eng, LdoParams
from p5v_design import analysis, schematic, plots, stability
from p5v_design import plots_wc_mc as pwc
from p5v_design.ldo_model import dropout_v, ignd_a, tj_c, psrr_db, vadj_typical_vs_temp
from dataclasses import replace

# --- EDITABLE BOM ---
# Datasheet TA01, "5V Supply with Shutdown" (LT3010/LT3010-5 Rev. J page 1).
# Fixed 5 V part: SENSE tied to OUT (R1 = 0). No external divider (R2 open).
# Capacitors are the 1 µF parts drawn on that figure. Load is the labeled 50 mA.
# VIN is printed as 5.4 V to 80 V; 12 V is the single-point bias inside that range.
R1 = 0.0             # SENSE shorted to OUT
R2 = 1.0e12          # open — no bottom resistor
CFF = 0.0            # no feedforward capacitor on the fixed part
COUT = 1.0e-6
CIN = 1.0e-6
ESR_OUT = 10e-3      # ceramic assumption, not printed on TA01
ESR_IN = 10e-3
RLOAD = 100.0        # 5 V / 50 mA
R_TOL = 0.01         # 1 %
C_TOL = 0.10         # 10 %
VIN_NOM = 12.0
VIN_MIN = 5.4
VIN_MAX = 80.0
ILOAD_OP = 50e-3
N_MC = 2000
N_SPICE_MC = 40
N_LTSPICE_MC = 200
SKIP_SPICE_WC_MC = False
RUN_LTSPICE_BATCH = False
SPICE_WORKERS = None
LTSPICE_WORKERS = None

p = CircuitParams(
    r1=R1, r2=R2, cff=CFF, cout=COUT, cin=CIN,
    esr_out=ESR_OUT, esr_in=ESR_IN, r_load=RLOAD,
    r_tol=ToleranceSpec(R_TOL), c_tol=ToleranceSpec(C_TOL),
)
p = replace(p, op=replace(p.op, vin_nom=VIN_NOM, vin_min=VIN_MIN, vin_max=VIN_MAX,
                           iload_op=ILOAD_OP, r_load=RLOAD))

FIG = ROOT / "results" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
print("gain =", p.gain, "  Vout nom =", analysis.vout_dc(p))
print("SPICE_WORKERS =", SPICE_WORKERS, "  N_MC =", N_MC)


gain = 1.0   Vout nom = 5.0
SPICE_WORKERS = None   N_MC = 2000


In [2]:
from p5v_design.supply_budget import rail_budget, budget_vs_load, budget_vs_vin
b = rail_budget(p)
bmax = rail_budget(p, which="max")
typ = analysis.metrics_table(b).rename(columns={"value": "typ"})
mx = analysis.metrics_table(bmax).rename(columns={"value": "max Ignd"})
op_tbl = typ.join(mx[["max Ignd"]])[["typ", "max Ignd", "unit"]]
display(analysis.style_metrics(op_tbl))
bl = budget_vs_load(p)
display(analysis.style_metrics(analysis.wide_metrics_table(bl, header="iload")))
bl.to_csv(ROOT/"results"/"supply_budget.csv", index=False)
ann = [dict(text="50 mA load", xy=(p.op.vin_nom, 100*b["efficiency"]), offset=(12, 10), arrow=True)]
plots.plot_efficiency(p, annotations=ann, path=FIG/"09_eta.png")
plt.show()
fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.plot(1e3*bl.iload, 1e3*bl.iin, "k-", lw=1.7, label="Iin")
ax.plot(1e3*bl.iload, 1e3*bl.iload, "C0--", lw=1.2, label="Iout")
ax.plot(1e3*bl.iload, 1e3*bl.ignd, "C3-.", lw=1.3, label="Ignd")
ax.set_xlabel("Iout (mA)")
ax.set_ylabel("mA")
ax.set_title("Current drawn from VIN")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plots.apply_annotations(ax, [dict(text="Ignd << Iout at 50 mA", xy=(50, 1.3), offset=(10, 12), arrow=True)])
plots.savefig(fig, FIG/"09_iin.png")
plt.show()
plots.plot_rail_stack(b, path=FIG/"09_rail_stack.png")
plt.show()
bv = budget_vs_vin(p)
bvmax = budget_vs_vin(p, which="max")
fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.plot(bv.vin, 1e3*bv.iin, "k-", lw=1.7, label="Iin typ")
ax.plot(bvmax.vin, 1e3*bvmax.iin, "C3--", lw=1.4, label="Iin max Ignd")
ax.set_xlabel("Vin (V)"); ax.set_ylabel("mA"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_title("Input current vs Vin (typ vs max Ignd)")
plots.savefig(fig, FIG/"09_iin_vs_vin.png"); plt.show()
fig, ax = plt.subplots(figsize=(8.0, 4.0))
ax.stackplot(bv.vin, bv.p_out, bv.p_diss, labels=["Pout", "Pdiss"], colors=["#2ca02c", "#ff7f0e"], alpha=0.85)
ax.set_xlabel("Vin (V)"); ax.set_ylabel("W"); ax.legend(fontsize=8)
ax.set_title("Pin = Pout + Pdiss")
ax.grid(True, alpha=0.3)
plots.savefig(fig, FIG/"09_pin_stack.png"); plt.show()
plots.plot_budget_grouped(b, bmax, path=FIG/"09_budget_grouped.png")
plt.show()
plots.plot_efficiency_vs_load(p, annotations=[
    dict(text="50 mA", xy=(50.0, 100*b["efficiency"]), offset=(12, 10), arrow=True),
], path=FIG/"09_eta_vs_load.png")
plt.show()
# Stacked current at 8 / 9 / 12 V (comparator 07 analog)
vins = [5.4, 12.0, 24.0]
iout = [1e3 * rail_budget(p, vin=v)["iload"] for v in vins]
ig = [1e3 * rail_budget(p, vin=v)["ignd"] for v in vins]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
x = np.arange(len(vins))
ax.bar(x, iout, color="C0", edgecolor="k", lw=0.4, label="Iout")
ax.bar(x, ig, bottom=iout, color="C3", edgecolor="k", lw=0.4, label="Ignd")
ax.set_xticks(x); ax.set_xticklabels([f"{v:g} V" for v in vins])
ax.set_ylabel("mA"); ax.set_title(r"$I_{\mathrm{IN}}=I_{\mathrm{OUT}}+I_{\mathrm{GND}}$")
ax.legend(fontsize=8); ax.grid(True, axis="y", alpha=0.3)
plots.savefig(fig, FIG/"09_iin_stacked.png"); plt.show()

,typ,max Ignd,unit
parameter,,,
vin,12,12,V
vout,5,5,V
iload,50,50,mA
ignd,1.8,3.3,mA
iin,51.8,53.3,mA
p_in,621.6,639.6,mW
p_out,250,250,mW
p_diss,371.6,389.6,mW
efficiency,40.2188,39.0869,%


,unit,1 mA,10 mA,50 mA
parameter,,,,
vin,V,12,12,12
vout,V,5,5,5
ignd,mA,0.1,0.4,1.8
iin,mA,1.1,10.4,51.8
p_in,mW,13.2,124.8,621.6
p_out,mW,5,50,250
p_diss,mW,8.2,74.8,371.6
efficiency,%,37.8788,40.0641,40.2188


cell:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
cell:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plots.savefig(fig, FIG/"09_iin_vs_vin.png"); plt.show()


cell:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plots.savefig(fig, FIG/"09_pin_stack.png"); plt.show()


cell:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:47: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


cell:59: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plots.savefig(fig, FIG/"09_iin_stacked.png"); plt.show()
